[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/48_ring_attention_solution.ipynb)

# 🔴 Solution: Ring Attention

Sequence-parallel attention via ring communication, simulated with online softmax.

Each device $i$ holds query chunk $Q_i$ and iterates over all key/value chunks in ring order, accumulating the output with the online softmax recurrence:

$$m^{(j)} = \max(m^{(j-1)},\, \max_k s_{ik}), \quad l^{(j)} = l^{(j-1)} e^{m^{(j-1)}-m^{(j)}} + \sum_k e^{s_{ik}-m^{(j)}}$$

$$O^{(j)} = \frac{O^{(j-1)}\, l^{(j-1)} e^{m^{(j-1)}-m^{(j)}} + e^{S-m^{(j)}} V_j}{l^{(j)}}$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def ring_attention(Q, K, V, num_devices):
    B, S, D = Q.shape
    chunk = S // num_devices
    scale = D ** -0.5

    Q_chunks = Q.split(chunk, dim=1)
    K_chunks = K.split(chunk, dim=1)
    V_chunks = V.split(chunk, dim=1)

    outputs = []
    for i, Qi in enumerate(Q_chunks):
        m = torch.full((B, chunk, 1), float('-inf'), device=Q.device, dtype=Q.dtype)
        l = torch.zeros(B, chunk, 1, device=Q.device, dtype=Q.dtype)
        o = torch.zeros(B, chunk, D, device=Q.device, dtype=Q.dtype)

        for j in range(num_devices):
            Kj = K_chunks[(i + j) % num_devices]
            Vj = V_chunks[(i + j) % num_devices]
            scores = (Qi @ Kj.transpose(-2, -1)) * scale
            m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
            exp_scores = torch.exp(scores - m_new)
            l_new = l * torch.exp(m - m_new) + exp_scores.sum(dim=-1, keepdim=True)
            o = o * (l * torch.exp(m - m_new)) / l_new + (exp_scores @ Vj) / l_new
            m, l = m_new, l_new

        outputs.append(o)

    return torch.cat(outputs, dim=1)

In [ ]:
# Verify numerical equivalence with standard attention for 2 and 4 devices
torch.manual_seed(42)
B, S, D = 2, 8, 16
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V

for num_devices in [2, 4]:
    ring_out = ring_attention(Q, K, V, num_devices=num_devices)
    max_diff = (ring_out - ref_out).abs().max().item()
    match = torch.allclose(ring_out, ref_out, atol=1e-5)
    print(f'num_devices={num_devices}  shape={tuple(ring_out.shape)}  max_diff={max_diff:.2e}  match={match}')

In [ ]:
from torch_judge import check
check('ring_attention')